# ✅ RAG Production Checklist

Before going live, make sure your RAG system handles real-world conditions.
This notebook walks through the most important things to check — with code.

```
□ 1. Input validation     — handle bad/empty queries
□ 2. Fallback answers     — what if no docs are found?
□ 3. Latency logging      — know how slow each step is
□ 4. Caching              — don't embed the same query twice
□ 5. Rate limiting        — protect against abuse
```

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import time

model = SentenceTransformer('all-MiniLM-L6-v2')

docs = [
    "RAG stands for Retrieval Augmented Generation.",
    "Embeddings convert text into vectors.",
    "Chunking splits documents into smaller pieces.",
]
doc_embeddings = model.encode(docs)
print("Ready!")

Ready!


## ✅ Check 1 — Input Validation

In [2]:
# Always validate the query before doing any work

def validate_query(question):
    if not question or not question.strip():
        return False, "Question cannot be empty"
    if len(question.strip()) < 3:
        return False, "Question too short"
    if len(question) > 1000:
        return False, "Question too long (max 1000 chars)"
    return True, None

test_inputs = ["", "hi", "What is RAG?" * 200, "What is RAG?"]

for q in test_inputs:
    ok, err = validate_query(q)
    label = repr(q[:30])
    print(f"{'✅' if ok else '❌'} {label:<35} {err or 'valid'} ")

❌ ''                                  Question cannot be empty 
❌ 'hi'                                Question too short 
❌ 'What is RAG?What is RAG?What i'    Question too long (max 1000 chars) 
✅ 'What is RAG?'                      valid 


## ✅ Check 2 — Fallback When No Docs Found

In [3]:
# If the top score is too low, the docs aren't relevant — don't hallucinate

MIN_RELEVANCE = 0.30  # tune this for your use case

def search_with_fallback(question):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = int(np.argmax(scores))
    top_score = float(scores[top_idx])

    if top_score < MIN_RELEVANCE:
        return None, top_score   # Signal: no good result
    return docs[top_idx], top_score


questions = [
    "What is RAG?",
    "What is the best pizza topping?",   # off-topic
]

for q in questions:
    result, score = search_with_fallback(q)
    if result:
        print(f"✅ [{score:.2f}] '{q}'")
        print(f"   → {result}")
    else:
        print(f"⚠️  [{score:.2f}] '{q}'")
        print(f"   → Sorry, I don't have information on that.")
    print()

✅ [0.78] 'What is RAG?'
   → RAG stands for Retrieval Augmented Generation.

⚠️  [0.05] 'What is the best pizza topping?'
   → Sorry, I don't have information on that.



## ✅ Check 3 — Latency Logging

In [4]:
# Know how long each step takes — so you know where to optimise

def timed_rag(question):
    timings = {}

    t0 = time.time()
    q_emb = model.encode(question)
    timings["embed_query"] = time.time() - t0

    t0 = time.time()
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:2]
    context = [docs[i] for i in top_idx]
    timings["search"] = time.time() - t0

    # LLM call would go here
    timings["llm"] = 0.0   # placeholder

    total = sum(timings.values())
    print(f"Question: '{question}'")
    print(f"  embed_query : {timings['embed_query']*1000:.1f}ms")
    print(f"  search      : {timings['search']*1000:.1f}ms")
    print(f"  llm         : {timings['llm']*1000:.1f}ms  (placeholder)")
    print(f"  TOTAL       : {total*1000:.1f}ms")
    return context

timed_rag("What is RAG?")

Question: 'What is RAG?'
  embed_query : 39.9ms
  search      : 0.8ms
  llm         : 0.0ms  (placeholder)
  TOTAL       : 40.7ms


['RAG stands for Retrieval Augmented Generation.',
 'Chunking splits documents into smaller pieces.']

## ✅ Check 4 — Simple Query Cache

In [5]:
# Identical queries should not re-embed and re-search
# A simple dict cache saves time in production

cache = {}  # question → (context, answer)

def cached_search(question):
    if question in cache:
        print(f"  Cache HIT  → '{question}'")
        return cache[question]

    print(f"  Cache MISS → '{question}' — running search...")
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:2]
    result = [docs[i] for i in top_idx]
    cache[question] = result
    return result

print("First call:")
cached_search("What is RAG?")
print("Second call (same question):")
cached_search("What is RAG?")
print(f"\nCache now holds {len(cache)} unique query/result pairs")

First call:
  Cache MISS → 'What is RAG?' — running search...
Second call (same question):
  Cache HIT  → 'What is RAG?'

Cache now holds 1 unique query/result pairs


## ✅ Check 5 — Rate Limiting

In [6]:
# Prevent a single user from hammering the API
from collections import defaultdict

MAX_REQUESTS_PER_MINUTE = 5
request_counts = defaultdict(list)  # user_id → [timestamps]

def is_rate_limited(user_id):
    now = time.time()
    window = 60  # 1 minute

    # Remove old timestamps outside the window
    request_counts[user_id] = [
        t for t in request_counts[user_id] if now - t < window
    ]

    if len(request_counts[user_id]) >= MAX_REQUESTS_PER_MINUTE:
        return True  # rate limited

    request_counts[user_id].append(now)
    return False


user = "alice"
for i in range(7):
    blocked = is_rate_limited(user)
    status  = "🚫 BLOCKED" if blocked else "✅ allowed"
    print(f"  Request {i+1}: {status}")

  Request 1: ✅ allowed
  Request 2: ✅ allowed
  Request 3: ✅ allowed
  Request 4: ✅ allowed
  Request 5: ✅ allowed
  Request 6: 🚫 BLOCKED
  Request 7: 🚫 BLOCKED


## Summary

| Check | Why It Matters |
|---|---|
| Input validation | Prevents crashes from bad input |
| Fallback answer | Avoids hallucination when docs aren't relevant |
| Latency logging | Shows you exactly what to optimise |
| Caching | Saves embedding cost on repeated queries |
| Rate limiting | Protects your API and LLM budget |